# Adversarial Temporal Tasks — Tests & Metrics

This notebook runs the **adversarial temporal benchmark**: Reversion, Interval, and Causal Reasoning tasks with task-specific metrics. See `ADVERSARIAL_TEMPORAL_TASKS.md` for the full taxonomy.

**Metrics:**
- **Reversion Accuracy** — Facts revert to a previous state (e.g. CEO: Alice → Bob → Alice); system must return the *current* state at query time.
- **Interval Accuracy** — Queries at validity-window boundaries and midpoints; system must respect `valid_from ≤ query_time < valid_to`.
- **Causal Reasoning Accuracy** — “Before X”, “after Y”, “between” style questions; requires ordered timeline reasoning.

In [ ]:
# @title ## Setup: Install Dependencies
!pip install -q pandas matplotlib
print("✓ Dependencies installed")

In [ ]:
# @title ## Setup: Clone Repository
REPO_URL = "https://github.com/YOUR_USERNAME/time.git"  # Set to your fork or repo
REPO_DIR = "/content/time"

import os
import shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
exit_code = os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
if exit_code != 0:
    print("Clone failed. Upload the repo zip and run: from google.colab import files; files.upload()")
    print("Then: !unzip -q -o time.zip -d /content && (mv /content/time-main /content/time 2>/dev/null || true)")
    raise SystemExit(1)
%cd /content/time
print("✓ Repository ready")

## 1. Generate Adversarial Data

Creates `benchmarks/adversarial_temporal_facts.jsonl` and `adversarial_temporal_questions.jsonl` with Reversion, Interval, and CausalReasoning questions.

In [ ]:
# @title Generate adversarial temporal facts and questions
%cd /content/time
!python scripts/generate_adversarial_temporal.py --out-dir benchmarks
print("✓ Adversarial data generated")

## 2. Run Adversarial Benchmark

Evaluates systems **A** (Plain RAG), **B** (Temporal Rerank), **C** (Time Constraint), **D** (TTA) on each task family and writes `results/adversarial_temporal_results.csv`.

In [ ]:
# @title Run benchmark by task type
%cd /content/time
!python scripts/run_adversarial_benchmark.py --facts benchmarks/adversarial_temporal_facts.jsonl --questions benchmarks/adversarial_temporal_questions.jsonl --out results/adversarial_temporal_results.csv --systems A,B,C,D
print("✓ Benchmark complete")

## 3. View Results Table

Per-system metrics: Reversion Accuracy, Interval Accuracy, Causal Reasoning Accuracy, Overall Accuracy.

In [ ]:
import pandas as pd

df = pd.read_csv("results/adversarial_temporal_results.csv")
display_cols = [c for c in df.columns if c.endswith("Accuracy") or c == "system"]
print("=== Adversarial Temporal Results ===")
display(df[display_cols])

## 4. Charts by Task Type

Bar charts: each metric by system. TTA (D) and Time Constraint (C) should lead on Reversion and Interval when intervals are respected.

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv("results/adversarial_temporal_results.csv")
systems = df["system"].tolist()
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4'][:len(systems)]

metrics = ["ReversionAccuracy", "IntervalAccuracy", "CausalReasoningAccuracy", "OverallAccuracy"]
metrics = [m for m in metrics if m in df.columns]
n = len(metrics)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for i, m in enumerate(metrics):
    ax = axes[i]
    ax.bar(systems, df[m] * 100, color=colors)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(m.replace("Accuracy", "").strip() or "Overall")
    ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig("results/adversarial_temporal_charts.png", dpi=150)
plt.show()
print("✓ Saved results/adversarial_temporal_charts.png")

## 5. Task Types & Expected Gaps

| Task | What it tests | Typical failure (Plain RAG) |
|------|----------------|-----------------------------|
| **Reversion** | State reverts (A→B→A); query after reversion must return A | Picks “latest” by timestamp and returns B |
| **Interval** | Validity windows; boundary/midpoint queries | Ignores interval, returns wrong window |
| **Causal** | Before/after/between in timeline | Pattern match instead of ordered timeline |

**Systems:** A = Plain RAG, B = Temporal Rerank, C = Time Constraint, D = TTA (truth intervals + decay).